In [0]:

import argparse
import logging
import os

import sys
sys.path.append("..")
sys.path.append("../..")

import lib_etl.validations_ETL as validations
from lib.job_manager import load_config, split_config
from lib.s3 import etl_input_table_validator

In [0]:
%run ../../config/utils

In [0]:
run_as_date = dbutils.widgets.get("run_as_date")

In [0]:

def main(data_paths, config_validation):
    logging.info("Starting processing table detail_isnr_fiscal")
    
    recency_lookback_duration = data_paths.get("recency_lookback_duration", {})
   
    etl_input_table_validator(
        silver_transaction_fiscal_detail,
        recency_lookback_duration=recency_lookback_duration,
        spark=spark
    )

    detail_fiscal = spark.table(silver_transaction_fiscal_detail)

    filter_in_store = detail_fiscal["SALES_CTGRY_CD"] == "03"
    filter_no_returns = detail_fiscal["SALES_QTY"] > 0
    # still saw returns after prior filter
    filter_no_returns_deli = detail_fiscal["QTY_IN_UNITS"] > 0

    filters = filter_in_store & filter_no_returns & filter_no_returns_deli

    detail_isnr_fiscal = detail_fiscal.filter(filters)
    detail_isnr_fiscal.createOrReplaceTempView('source')

    validations.validate_table(
        spark,
        "intermediate",
        "detail_isnr_fiscal",
        config_validation,
        detail_isnr_fiscal,
        stats_etl_path
    )

    logging.info(
        "Saving the intermediate file "
        + silver_transaction_fiscal_detail_isnr
    )

    detail_isnr_fiscal.write.mode("overwrite").saveAsTable(silver_transaction_fiscal_detail_isnr)

    if archive_flag:
        save_archive(detail_isnr_fiscal, silver_transaction_fiscal_detail_isnr_archive, run_as_date)
    # (detail_isnr_fiscal.repartition("FISCAL_WEEK_END")
    #  .write
    #  .format("delta")
    #  .mode("overwrite")
    #  .partitionBy("FISCAL_WEEK_END")
    #  .saveAsTable(silver_transaction_fiscal_detail_isnr))

    print("detail_fiscal schema")
    detail_fiscal.printSchema()
    print("detail_isnr_fiscal schema")
    detail_isnr_fiscal.printSchema()


In [0]:

parser = argparse.ArgumentParser()

try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()

default_config_path = os.path.join(base_dir, "../config/config.yaml")

print(f"base_dir: {base_dir}")
print(f"default_config_path: {default_config_path}")

parser.add_argument(
   "--config_path",
    type=str,
    default=default_config_path,
    help=(
        """
        path to the config file
        """
    ),
)

try:
    print("Parsing args")
    args, unknown = parser.parse_known_args()

    print("Loading config")
    config = load_config(args.config_path)

    print("Splitting config")
    data_paths, club_square_config, config_validation = split_config(config)

    print("Running main")
    main(data_paths, config_validation)

except Exception as e:
    print(e)
